# __Activity 5, Part 1__
## Rachel Nana, January 2019

__Importing Packages__

In [ ]:
import pandas as pd
import sklearn
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

import pickle

__Importing Telco Data__

In [ ]:
telco = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv');

In [ ]:
telco.head()

In [ ]:
telco.shape

In [ ]:
(m, n) = telco.shape

__Write Random Subset to CSV__

In [ ]:
rand_sample = telco.sample(n=round(m/4), replace=False)
rand_sample.drop('Churn', axis=1, inplace=True)

In [ ]:
telco.head()

In [ ]:
rand_sample.head()

In [ ]:
rand_sample.shape

In [ ]:
rand_sample.to_csv('./telco_random_sample.csv', index=False)

__Data Prep__

Variables

In [ ]:
X = telco.loc[:, telco.columns != 'Churn'].copy()
y = telco.loc[:, 'Churn'].copy()

Split The Data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
X_train.head()

In [ ]:
y_train.head()

In [ ]:
X_train.shape

In [ ]:
from data_cleansing import TelcoPreprocessing
(X_train, y_train) = TelcoPreprocessing(X_train, y_train, X_only=False)

Pipeline

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

RandForest = RandomForestClassifier()
SimpleImputer = SimpleImputer()

In [ ]:
pipe = Pipeline([('SI', SimpleImputer), 
                 ('RF', RandForest)])

In [ ]:
pipe.set_params(SI__missing_values=np.nan, SI__strategy="median",
                RF__criterion="gini", RF__max_depth=9, RF__n_estimators=200)

Train The Model

In [ ]:
pipe.fit(X_train, y_train)

__Model Validation__

Predictions

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score, log_loss
from matplotlib import pyplot as plt

In [ ]:
(X_test, y_test) = TelcoPreprocessing(X_test, y_test, X_only=False)

In [ ]:
pred_prob = pipe.predict_proba(X_test)
roc_score = roc_auc_score(y_test, pred_prob[:,1])
print("ROC Score: {}".format(roc_score))

In [ ]:
pipe.score(X_test, y_test)

In [ ]:
f = plt.figure()
fpr, tpr, _ = roc_curve(y_test, pred_prob[:,1])
plt.plot(fpr, tpr, color='blue', label='Random Forest (AUC = %0.4f)' % roc_score)
plt.plot([0,1],[0,1], color = 'orange', label = 'No Information')
plt.legend(loc="lower right")
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC curve (Random Forest Pipeline)')
plt.show()

f.savefig("ROC_curve_RF_Pipe.pdf", bbox_inches='tight')

__Retrain on Full Dataset__

In [ ]:
(X_pre, y_pre) = TelcoPreprocessing(X, y, X_only=False)

In [ ]:
pipe2 = Pipeline([('SI', SimpleImputer), 
                 ('RF', RandForest)])

In [ ]:
pipe2.set_params(SI__missing_values=np.nan, SI__strategy="median",
                RF__criterion="gini", RF__max_depth=9, RF__n_estimators=200)

In [ ]:
pipe2.fit(X_pre, y_pre)

__Pickle Save__

In [ ]:
def pickle_save(name, item):
    PIK = str(name)+ ".pickle"
    with open(PIK,"wb") as f:
        pickle.dump(item, f)

In [ ]:
pickle_save('telco_pipeline', pipe2)